In [1]:
from IPython.display import display, Markdown

In [2]:
import sys
import pandas as pd

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("Interpreter:", sys.executable)

Python: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]
Pandas: 3.0.6
Interpreter: d:\Coding\Data_Portfolio\ecommerce-analytics\.venv\Scripts\python.exe


In [3]:
from pathlib import Path

project_root = Path.cwd().parent
file_path = project_root / "data" / "raw" / "olist_orders_dataset.csv"

print(file_path)
print("File exists:", file_path.exists())

d:\Coding\Data_Portfolio\ecommerce-analytics\data\raw\olist_orders_dataset.csv
File exists: True


In [4]:
orders = pd.read_csv(file_path)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [6]:
# Calculate order_id validation metrics
order_id_validation = {
    "Total rows": len(orders),
    "Unique IDs": orders["order_id"].nunique(),
    "Duplicate IDs": orders["order_id"].duplicated().sum(),
    "Missing IDs": orders["order_id"].isna().sum()
}

# Display results
validation_df = pd.DataFrame(
    order_id_validation.items(),
    columns=["Check", "Result"]
)

display(validation_df.style.format({"Result": "{:,}"}))

,Check,Result
0,Total rows,"99,441"
1,Unique IDs,"99,441"
2,Duplicate IDs,0
3,Missing IDs,0


In [7]:
columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for col in columns:
    missing = orders[col].isna().sum()
    print(f"- {col}: {missing:,} missing values")

- order_approved_at: 160 missing values
- order_delivered_carrier_date: 1,783 missing values
- order_delivered_customer_date: 2,965 missing values


In [8]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [9]:
orders.loc[orders["order_delivered_customer_date"].isna(), "order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [10]:
orders.loc[
    (orders["order_status"] == "delivered") & 
    (orders["order_delivered_customer_date"].isna())
    ]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [11]:
orders.loc[
    (orders["order_status"] != "delivered") &
    (orders["order_delivered_customer_date"].notna()),
    "order_status"].value_counts()

order_status
canceled    6
Name: count, dtype: int64

In [12]:
orders.loc[
    (orders["order_status"] != "delivered") &
    (orders["order_delivered_customer_date"].notna())
    ]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


In [13]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

In [14]:
orders["order_purchase_timestamp"].dtype

dtype('<M8[us]')

In [15]:
date_columns = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

In [16]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [17]:
delivery_before_purchase = orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
print("Order that delivered before purchase:", delivery_before_purchase.sum())

Order that delivered before purchase: 0


In [18]:
carrier_before_purchase = orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
print("Order that reached carrier before purchase:", carrier_before_purchase.sum())
print("Anomaly rate (%):", round(carrier_before_purchase.mean() * 100, 3))

Order that reached carrier before purchase: 166
Anomaly rate (%): 0.167


In [19]:
print(round(100.0 * carrier_before_purchase.mean(), 3), "%", sep="")

0.167%


In [20]:
carrier_anomalies = orders.loc[
    orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
].copy()
carrier_anomalies["time_difference"] = carrier_anomalies["order_delivered_carrier_date"] - carrier_anomalies["order_purchase_timestamp"]
carrier_anomalies["time_difference"].describe()

count                         166
mean     -2 days +21:57:37.879519
std       13 days 06:55:25.842359
min           -172 days +18:54:07
25%             -1 days +22:58:57
50%      -1 days +23:24:00.500000
75%             -1 days +23:45:56
max             -1 days +23:59:36
Name: time_difference, dtype: object

In [21]:
carrier_anomalies.loc[carrier_anomalies["time_difference"].idxmin()]

order_id                         7c48bb55e8e4f7e56d412e9653db37bc
customer_id                      34ef6181341eb36c47fd601c46878f00
order_status                                            delivered
order_purchase_timestamp                      2018-07-16 18:40:53
order_approved_at                             2018-07-16 18:50:22
order_delivered_carrier_date                  2018-01-26 13:35:00
order_delivered_customer_date                 2018-07-23 20:04:45
order_estimated_delivery_date                 2018-08-07 00:00:00
time_difference                               -172 days +18:54:07
Name: 25883, dtype: object

In [22]:
approved_before_purchased = orders["order_approved_at"] < orders["order_purchase_timestamp"]
print(approved_before_purchased.sum())

0


In [23]:
carrier_before_approval = orders["order_delivered_carrier_date"] < orders["order_approved_at"]

print("Orders that reached carrier before approval:", carrier_before_approval.sum())
print("Anomaly rate (%): ", round(carrier_before_approval.mean() * 100, 3))

Orders that reached carrier before approval: 1359
Anomaly rate (%):  1.367


In [24]:
delivery_before_carrier = orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
print("Orders delivered before carrier:", delivery_before_carrier.sum())

Orders delivered before carrier: 23


In [25]:
# 1. Define chronology validation rules
approval_before_purchase = (
    orders["order_approved_at"] < orders["order_purchase_timestamp"]
)

carrier_before_purchase = (
    orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
)

delivery_before_purchase = (
    orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
)

carrier_before_approval = (
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
)

delivery_before_carrier = (
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
)

# 2. Combine rules using OR
chronology_anomaly_mask = (
    approval_before_purchase
    | carrier_before_purchase
    | delivery_before_purchase
    | carrier_before_approval
    | delivery_before_carrier
)

# 3. Extract affected orders
chronology_anomalies = orders.loc[chronology_anomaly_mask].copy()

# 4. Summarize findings
print("Total orders:", len(orders))
print("Orders with chronology anomalies:", chronology_anomaly_mask.sum())
print("Anomaly rate (%):", round(chronology_anomaly_mask.mean() * 100, 3))

Total orders: 99441
Orders with chronology anomalies: 1382
Anomaly rate (%): 1.39


## 2. Customers Dataset Profiling

In [26]:
customers_path = project_root / "data" / "raw" / "olist_customers_dataset.csv"

customers = pd.read_csv(customers_path)

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [27]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [28]:
# Validate customers identifiers
customer_id_columns = ["customer_id", "customer_unique_id"]

customer_key_profile = pd.DataFrame({
    "unique_values": customers[customer_id_columns].nunique(),
    "missing_values": customers[customer_id_columns].isna().sum(),
    "duplicated_values": customers[customer_id_columns].apply(
        lambda column: column.duplicated().sum()
    ),
})

customer_key_profile

,unique_values,missing_values,duplicated_values
customer_id,99441,0,0
customer_unique_id,96096,0,3345


In [29]:
# Valudate the orders-to-customers relationship
order_customer_ids = set(orders["customer_id"].dropna())
customer_ids = set(customers["customer_id"].dropna())

unmatched_customer_ids = order_customer_ids - customer_ids

relationship_validation = {
    "total_orders": len(orders),
    "unique_order_customer_ids": len(order_customer_ids),
    "unique_customer_ids": len(customer_ids),
    "unmatched_customer_ids": len(unmatched_customer_ids)
}

pd.Series(relationship_validation)

total_orders                 99441
unique_order_customer_ids    99441
unique_customer_ids          99441
unmatched_customer_ids           0
dtype: int64

In [30]:
# Count distinct customer locations per unique customer
customer_locations = (
    customers.groupby("customer_unique_id")
    .agg(
        unique_cities=("customer_city", "nunique"),
        unique_states=("customer_state", "nunique"),
        unique_zip_prefixes=("customer_zip_code_prefix", "nunique"),
    )
)
# Identify customers associated with multiple postal code prefixes
customers_multiple_locations = customer_locations.loc[
    customer_locations["unique_zip_prefixes"] > 1
]

print("Customers with multiple ZIP prefixes:", len(customers_multiple_locations))


Customers with multiple ZIP prefixes: 250


## 3. Order Items Dataset Profiling

In [31]:
order_items_path = project_root / "data" / "raw" / "olist_order_items_dataset.csv"

order_items = pd.read_csv(order_items_path)

order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [32]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [33]:
# Validate the candidate composite primary key
order_item_key = ["order_id", "order_item_id"]

order_item_key_validation = pd.Series({
    "total_rows": len(order_items),
    "unique_orders": order_items["order_id"].nunique(),
    "unique_order_item_keys": (
        order_items[order_item_key].drop_duplicates().shape[0]
    ),
    "duplicate_composite_keys": (
        order_items.duplicated(subset=order_item_key).sum()
    ),
    "missing_key_values": (
        order_items[order_item_key].isna().any(axis=1).sum()
    ),
})

order_item_key_validation

total_rows                  112650
unique_orders                98666
unique_order_item_keys      112650
duplicate_composite_keys         0
missing_key_values               0
dtype: int64

In [34]:
# Validate referential integrity between orders and order items
order_ids = set(orders["order_id"].dropna())
order_item_order_ids = set(order_items["order_id"].dropna())

orders_without_items = order_ids - order_item_order_ids
items_without_orders = order_item_order_ids - order_ids

order_items_relationship = pd.Series({
    "total_orders": len(order_ids),
    "orders_with_items": len(order_ids & order_item_order_ids),
    "orders_without_items": len(orders_without_items),
    "item_order_ids_without_orders": len(items_without_orders),
})

order_items_relationship

total_orders                     99441
orders_with_items                98666
orders_without_items               775
item_order_ids_without_orders        0
dtype: int64

In [35]:
# Investigate order stasuses for orders without items
orders_without_items_df = orders.loc[
    orders["order_id"].isin(orders_without_items)
].copy()

orders_without_items_profile = (
    orders_without_items_df["order_status"]
    .value_counts()
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

orders_without_items_profile

,order_status,order_count
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


In [36]:
# Count item records per order
items_per_order = (
    order_items.groupby("order_id")
    .size()
    .rename("item_count")
)

# Summarize the distribution
items_per_order.describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: item_count, dtype: float64

In [37]:
# Join orders with their order items
orders_items_joined = orders.merge(
    order_items,
    on="order_id",
    how="left",
    validate="one_to_many",
    indicator=True,
)

# Compare row counts before and after the join
join_validation = pd.Series({
    "orders_before_join": len(orders),
    "rows_after_join": len(orders_items_joined),
    "distinct_orders_after_join": orders_items_joined["order_id"].nunique(),
    "orders_without_items": (
        orders_items_joined["_merge"].eq("left_only").sum()
    ),
})

join_validation

orders_before_join             99441
rows_after_join               113425
distinct_orders_after_join     99441
orders_without_items             775
dtype: int64

## 4. Order Payments Dataset Profiling

In [38]:
# Load the order payments dataset
order_payments_path = (
    project_root / "data" / "raw" / "olist_order_payments_dataset.csv"
)

order_payments = pd.read_csv(order_payments_path)

order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [39]:
# Inspect row count, data types, and missing values
order_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB


In [40]:
# Validate the candidate composite primary key
payment_key = ["order_id", "payment_sequential"]

payment_key_validation = pd.Series({
    "total_rows": len(order_payments),
    "unique_orders": order_payments["order_id"].nunique(),
    "unique_payment_keys": (
        order_payments[payment_key].drop_duplicates().shape[0]
    ),
    "duplicate_composite_keys": (
        order_payments.duplicated(subset=payment_key).sum()
    ),
    "missing_key_values": (
        order_payments[payment_key].isna().any(axis=1).sum()
    ),
})

payment_key_validation

total_rows                  103886
unique_orders                99440
unique_payment_keys         103886
duplicate_composite_keys         0
missing_key_values               0
dtype: int64

In [41]:
# Validate referential integrity between orders and payments
order_ids = set(orders["order_id"].dropna())
payment_order_ids = set(order_payments["order_id"].dropna())

orders_without_payments = order_ids - payment_order_ids
payments_without_orders = payment_order_ids - order_ids

payment_relationship_validation = pd.Series({
    "total_orders": len(order_ids),
    "orders_with_payments": len(order_ids & payment_order_ids),
    "orders_without_payments": len(orders_without_payments),
    "payments_without_orders": len(payments_without_orders),
})

payment_relationship_validation

total_orders               99441
orders_with_payments       99440
orders_without_payments        1
payments_without_orders        0
dtype: int64

In [42]:
# Investigate orders without payment records
orders_without_payments_df = orders.loc[
    orders["order_id"].isin(orders_without_payments)
].copy()

orders_without_payments_df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [43]:
# Count payment records per order
payments_per_order = (
    order_payments.groupby("order_id")
    .size()
    .rename("payment_count")
)

# Summarize the distribution
payments_per_order.describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
Name: payment_count, dtype: float64

In [44]:
# Analyze the distribution of payment records per order
payment_count_distribution = (
    payments_per_order
    .value_counts()
    .sort_index()
    .rename_axis("payment_count")
    .reset_index(name="order_count")
)

payment_count_distribution

,payment_count,order_count
0,1,96479
1,2,2382
2,3,301
3,4,108
4,5,52
5,6,36
6,7,28
7,8,11
8,9,9
9,10,5


In [45]:
# Identify the order with the highest number of payment records
max_payment_order_id = payments_per_order.idxmax()

# Retrieve all payment records for that order
max_payment_order = order_payments.loc[
    order_payments["order_id"] == max_payment_order_id
].sort_values("payment_sequential")

max_payment_order

,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86


In [46]:
# Summarize the order with the most payment entries
max_payment_summary = pd.Series({
    "payment_count": len(max_payment_order),
    "total_payment_value": max_payment_order["payment_value"].sum(),
    "payment_types": max_payment_order["payment_type"].nunique(),
    "zero_value_payments": max_payment_order["payment_value"].eq(0).sum(),
    "negative_value_payments": max_payment_order["payment_value"].lt(0).sum(),
})

max_payment_summary

payment_count               29.00
total_payment_value        457.99
payment_types                1.00
zero_value_payments          2.00
negative_value_payments      0.00
dtype: float64

In [47]:
# Calculate the original payment total
original_payment_total = order_payments["payment_value"].sum()

## Join order items with payment records
items_payments_joined = order_items.merge(
    order_payments,
    on="order_id",
    how="inner",
    validate="many_to_many",
)

# Compare results before and after joining
fanout_validation = pd.Series({
    "original_payment_total": original_payment_total,
    "joined_payment_total": items_payments_joined["payment_value"].sum(),
    "original_payment_records": len(order_payments),
    "joined_records": len(items_payments_joined),
})

fanout_validation

original_payment_total      16008872.12
joined_payment_total        20308134.71
original_payment_records      103886.00
joined_records                117601.00
dtype: float64

In [48]:
# Compare payment totals using the same population of orders
payments_with_items = order_payments.loc[
    order_payments["order_id"].isin(order_items["order_id"])
]

# Calculate payment inflation caused by join fanout
fanout_impact = pd.Series({
    "payment_total_matching_orders": payments_with_items["payment_value"].sum(),
    "payment_total_after_join": items_payments_joined["payment_value"].sum(),
})

fanout_impact["inflated_amount"] = (
    fanout_impact["payment_total_after_join"]
    - fanout_impact["payment_total_matching_orders"]
)

fanout_impact["inflation_pct"] = (
    fanout_impact["inflated_amount"]
    / fanout_impact["payment_total_matching_orders"]
    * 100
)

fanout_impact.round(2)

payment_total_matching_orders    15846280.17
payment_total_after_join         20308134.71
inflated_amount                   4461854.54
inflation_pct                          28.16
dtype: float64

In [49]:
payments_by_order = (
    order_payments.groupby("order_id", as_index=False)
    .agg(total_payment=("payment_value", "sum"))
)

payments_by_order.head()

,order_id,total_payment
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [50]:
# Validate payment aggregation
payment_aggregation_validation = pd.Series({
    "source_payment_total": order_payments["payment_value"].sum(),
    "aggregated_payment_total": payments_by_order["total_payment"].sum(),
    "aggregated_order_count": len(payments_by_order),
    "duplicate_order_ids": payments_by_order["order_id"].duplicated().sum(),
})

payment_aggregation_validation.round(2)

source_payment_total        16008872.12
aggregated_payment_total    16008872.12
aggregated_order_count         99440.00
duplicate_order_ids                0.00
dtype: float64

In [51]:
# Aggregate item records to one row per order
items_by_order = (
    order_items.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "size"),
        total_item_value=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
    )
)

items_by_order.head()

,order_id,item_count,total_item_value,total_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


In [52]:
# Join order-level item and payment summaries
order_financials = items_by_order.merge(
    payments_by_order,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

order_financials.head()

,order_id,item_count,total_item_value,total_freight_value,total_payment,_merge
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,72.19,both
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,259.83,both
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,216.87,both
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,25.78,both
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,218.04,both


In [53]:
# Validate the order-level join
order_financials_validation = pd.Series({
    "orders_with_items": len(items_by_order),
    "rows_after_join": len(order_financials),
    "duplicate_order_ids": order_financials["order_id"].duplicated().sum(),
    "unmatched_payments": order_financials["_merge"].eq("left_only").sum(),
    "source_item_total": items_by_order["total_item_value"].sum(),
    "joined_item_total": order_financials["total_item_value"].sum(),
})

order_financials_validation.round(2)

orders_with_items         98666.0
rows_after_join           98666.0
duplicate_order_ids           0.0
unmatched_payments            1.0
source_item_total      13591643.7
joined_item_total      13591643.7
dtype: float64

In [54]:
# Validate payment totals for orders with items
payment_reconciliation = pd.Series({
    "source_payment_total": payments_with_items["payment_value"].sum(),
    "joined_payment_total": order_financials["total_payment"].sum(),
    "orders_with_missing_payments": order_financials["total_payment"].isna().sum(),
})

# Calculate the difference between totals
payment_reconciliation["payment_difference"] = (
    payment_reconciliation["joined_payment_total"]
    - payment_reconciliation["source_payment_total"]
)

payment_reconciliation.round(2)

source_payment_total            15846280.17
joined_payment_total            15846280.17
orders_with_missing_payments           1.00
payment_difference                     0.00
dtype: float64

## 5. Order Reviews Dataset Profiling

In [55]:
# Load the order reviews dataset
order_reviews_path = (
    project_root / "data" / "raw" / "olist_order_reviews_dataset.csv"
)

order_reviews = pd.read_csv(order_reviews_path)

order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [56]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [57]:
# Validate review identifiers
review_key_columns = ["review_id", "order_id"]

review_key_profile = pd.DataFrame({
    "unique_values": order_reviews[review_key_columns].nunique(),
    "missing_values": order_reviews[review_key_columns].isna().sum(),
    "duplicate_rows": order_reviews[review_key_columns].apply(
        lambda column: column.duplicated().sum()
    ),
})

review_key_profile

,unique_values,missing_values,duplicate_rows
review_id,98410,0,814
order_id,98673,0,551


In [58]:
# Validate the candidate composite key
review_composite_key = ["review_id", "order_id"]

review_composite_key_validation = pd.Series({
    "total_rows": len(order_reviews),
    "unique_composite_keys": (
        order_reviews[review_composite_key]
        .drop_duplicates()
        .shape[0]
    ),
    "duplicate_composite_keys": (
        order_reviews.duplicated(subset=review_composite_key).sum()
    ),
    "missing_key_values": (
        order_reviews[review_composite_key].isna().any(axis=1).sum()
    ),
})

review_composite_key_validation

total_rows                  99224
unique_composite_keys       99224
duplicate_composite_keys        0
missing_key_values              0
dtype: int64

In [59]:
# Identify review IDs associated with multiple records
review_id_counts = (
    order_reviews.groupby("review_id")
    .size()
    .rename("record_count")
)

repeated_review_ids = review_id_counts.loc[
    review_id_counts > 1
]

# Summarized repeated review identifiers
repeated_review_summary = pd.Series({
    "review_ids_with_multiple_records": len(repeated_review_ids),
    "records_in_repeated_review_groups": repeated_review_ids.sum(),
    "maximum_records_per_review_id": repeated_review_ids.max(),
})

repeated_review_summary

review_ids_with_multiple_records      789
records_in_repeated_review_groups    1603
maximum_records_per_review_id           3
dtype: int64

In [60]:
# Identify the review ID with the most records
most_repeated_review_id = review_id_counts.idxmax()

# Retrieve all records associated with that review ID
most_repeated_review = (
    order_reviews.loc[
        order_reviews["review_id"] == most_repeated_review_id
    ]
    .sort_values("order_id")
)

most_repeated_review

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
39950,08528f70f579f0c830189efc523d2182,03310aa823a66056268a3bab36e827fb,1,Produto errado,Entrega do produto diferente do solicitado\r\n...,2018-08-03 00:00:00,2018-08-06 00:09:52
74820,08528f70f579f0c830189efc523d2182,53c71d3953507c6239ff73917ed358c9,1,Produto errado,Entrega do produto diferente do solicitado\r\n...,2018-08-03 00:00:00,2018-08-06 00:09:52
8404,08528f70f579f0c830189efc523d2182,7813842ae95e8c497fc0233232ae815a,1,Produto errado,Entrega do produto diferente do solicitado\r\n...,2018-08-03 00:00:00,2018-08-06 00:09:52


In [61]:
# Check consistency of review attributes for repeated review IDs
review_attributes = [
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
]

repeated_review_records = order_reviews.loc[
    order_reviews["review_id"].isin(repeated_review_ids.index)
]

review_consistency = (
    repeated_review_records.groupby("review_id")[review_attributes]
    .nunique(dropna=False)
)

# Identify review IDs with differences in any attribute
inconsistent_review_ids = review_consistency.loc[
    review_consistency.gt(1).any(axis=1)
]

review_consistency_summary = pd.Series({
    "repeated_review_ids": len(repeated_review_ids),
    "consistent_review_ids": (
        len(review_consistency) - len(inconsistent_review_ids)
    ),
    "inconsistent_review_ids": len(inconsistent_review_ids),
})

review_consistency_summary

repeated_review_ids        789
consistent_review_ids      789
inconsistent_review_ids      0
dtype: int64

In [62]:
# Count review records associated with each other
reviews_per_order = (
    order_reviews.groupby("order_id")
    .size()
    .rename("review_count")
)

# Identify orders with multiple review records
orders_with_multiple_reviews = reviews_per_order.loc[
    reviews_per_order > 1
]

# Summarize orders with multiple reviews
multiple_reviews_summary = pd.Series({
    "orders_with_multiple_reviews": len(orders_with_multiple_reviews),
    "records_in_multiple_review_groups": orders_with_multiple_reviews.sum(),
    "maximum_reviews_per_order": orders_with_multiple_reviews.max(),
})

multiple_reviews_summary

orders_with_multiple_reviews          547
records_in_multiple_review_groups    1098
maximum_reviews_per_order               3
dtype: int64

In [63]:
# Identify orders with multiple review records
multiple_review_records = order_reviews.loc[
    order_reviews["order_id"].isin(orders_with_multiple_reviews.index)
].copy()

# Check whether review scores differ within each other
review_score_consistency = (
    multiple_review_records.groupby("order_id")
    .agg(
        review_count=("review_id", "size"),
        unique_review_ids=("review_id", "nunique"),
        unique_review_scores=("review_score", "nunique"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max"),
    )
)

# Summarize score consistency across orders
review_score_validation = pd.Series({
    "orders_with_multiple_reviews": len(review_score_consistency),
    "orders_with_identical_scores": (
        review_score_consistency["unique_review_scores"].eq(1).sum()
    ),
    "orders_with_different_scores": (
        review_score_consistency["unique_review_scores"].gt(1).sum()
    ),
    "orders_with_multiple_review_ids": (
        review_score_consistency["unique_review_ids"].gt(1).sum()
    ),
})

review_score_validation


orders_with_multiple_reviews       547
orders_with_identical_scores       345
orders_with_different_scores       202
orders_with_multiple_review_ids    547
dtype: int64

In [64]:
# Identify orders with conflicting review scores
conflicting_order_ids = review_score_consistency.index[
    review_score_consistency["unique_review_scores"] > 1
]

# Retrieve the corresponding review records
conflicting_review_records = order_reviews.loc[
    order_reviews["order_id"].isin(conflicting_order_ids)
].copy()

# Convert review timestamps to datetime
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp",
]

for column in review_date_columns:
    conflicting_review_records[column] = pd.to_datetime(
        conflicting_review_records[column]
    )

# Sort records to inspect review history within each order
conflicting_review_records = conflicting_review_records.sort_values(
    ["order_id", "review_creation_date", "review_answer_timestamp"]
)

conflicting_review_records.head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21,2018-03-22 01:32:08
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29,2018-03-30 03:16:19
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14,2017-07-17 13:58:06
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19,2017-07-20 12:06:11
69438,405eb2ea45e1dbe2662541ae5b47e2aa,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Seria ótimo se tivesem entregue os 3 (três) pe...,2018-03-06,2018-03-06 19:50:32
8273,b04ed893318da5b863e878cd3d0511df,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Um ponto negativo que achei foi a cobrança de ...,2018-03-20,2018-03-21 02:28:23
51527,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,03c939fd7fd3b38f8485a0f95798f1f6,4,NaN,NaN,2018-03-29,2018-03-30 00:29:09
88624,36ce47fb903bec726f89c65eac26dc9f,03eba6d9fef8f5b3e811d4b5a7cca9cd,4,NaN,NaN,2018-02-23,2018-02-23 23:55:53


In [65]:
# Summarize review timestamp variation within conflicting-score orders
review_timestamp_profile = (
    conflicting_review_records.groupby("order_id")
    .agg(
        review_count=("review_id", "size"),
        unique_creation_dates=("review_creation_date", "nunique"),
        unique_answer_timestamps=("review_answer_timestamp", "nunique"),
        first_review_date=("review_creation_date", "min"),
        last_review_date=("review_creation_date", "max"),
    )
)

# Calculate the time between first and last review creation dates
review_timestamp_profile["review_date_span_days"] = (
    review_timestamp_profile["last_review_date"]
    - review_timestamp_profile["first_review_date"]
).dt.days

# Summarize the results
review_timestamp_validation = pd.Series({
    "orders_with_conflicting_scores": len(review_timestamp_profile),
    "orders_with_different_creation_dates": (
        review_timestamp_profile["unique_creation_dates"].gt(1).sum()
    ),
    "orders_with_same_creation_date": (
        review_timestamp_profile["unique_creation_dates"].eq(1).sum()
    ),
    "median_review_date_span_days": (
        review_timestamp_profile["review_date_span_days"].median()
    ),
    "maximum_review_date_span_days": (
        review_timestamp_profile["review_date_span_days"].max()
    ),
})

review_timestamp_validation

orders_with_conflicting_scores          202.0
orders_with_different_creation_dates    172.0
orders_with_same_creation_date           30.0
median_review_date_span_days              5.0
maximum_review_date_span_days            88.0
dtype: float64

In [66]:
# Validate referential integrity between orders and reviews
order_ids = set(orders["order_id"].dropna())
review_order_ids = set(order_reviews["order_id"].dropna())

orders_without_reviews = order_ids - review_order_ids
reviews_without_orders = review_order_ids - order_ids 

review_relationship_validation = pd.Series({
    "total_orders": len(order_ids),
    "orders_with_reviews": len(order_ids & review_order_ids),
    "orders_without_reviews": len(orders_without_reviews),
    "review_order_ids_without_orders": len(reviews_without_orders),
})

review_relationship_validation

total_orders                       99441
orders_with_reviews                98673
orders_without_reviews               768
review_order_ids_without_orders        0
dtype: int64

In [67]:
# Validate the distribution and allowed range of review scores
review_score_distribution = (
    order_reviews["review_score"]
    .value_counts()
    .sort_index()
    .rename_axis("review_score")
    .reset_index(name="review_count")
)

review_score_distribution

,review_score,review_count
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


In [68]:
# Identify missing or out-of-range review scores
invalid_review_score_mask = (
    order_reviews["review_score"].isna()
    | ~order_reviews["review_score"].between(1, 5)
)

review_score_validation = pd.Series({
    "total_review_records": len(order_reviews),
    "missing_review_scores": order_reviews["review_score"].isna().sum(),
    "invalid_review_scores": invalid_review_score_mask.sum(),
    "minimum_review_score": order_reviews["review_score"].min(),
    "maximum_review_score": order_reviews["review_score"].max(),
})

review_score_validation

total_review_records     99224
missing_review_scores        0
invalid_review_scores        0
minimum_review_score         1
maximum_review_score         5
dtype: int64

In [69]:
# Investigate review coverage by order status
review_coverage = orders.assign(
    has_review=orders["order_id"].isin(review_order_ids)
)

review_coverage_by_status = (
    review_coverage.groupby("order_status")
    .agg(
        total_orders=("order_id", "size"),
        orders_with_reviews=("has_review", "sum"),
    )
)

review_coverage_by_status["orders_without_reviews"] = (
    review_coverage_by_status["total_orders"]
    - review_coverage_by_status["orders_with_reviews"]
)

review_coverage_by_status["review_coverage_pct"] = (
    review_coverage_by_status["orders_with_reviews"]
    / review_coverage_by_status["total_orders"]
    * 100
)

review_coverage_by_status.round(2)

,total_orders,orders_with_reviews,orders_without_reviews,review_coverage_pct
order_status,,,,
approved,2,2,0,100.00
canceled,625,605,20,96.80
created,5,3,2,60.00
delivered,96478,95832,646,99.33
invoiced,314,309,5,98.41
processing,301,295,6,98.01
shipped,1107,1032,75,93.22
unavailable,609,595,14,97.70


## 6. Geolocation Dataset Profiling

In [70]:
# Load the geolocation dataset
geolocation_path = (
    project_root / "data" / "raw" / "olist_geolocation_dataset.csv"
)

geolocation = pd.read_csv(geolocation_path)

geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [71]:
# Inspect row count, column types, and missing values
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


In [72]:
# Profile geolocation identifiers and duplicates
geolocation_key_profile = pd.Series({
    "total_rows": len(geolocation),
    "unique_zip_prefixes": geolocation["geolocation_zip_code_prefix"].nunique(),
    "duplicate_zip_prefix_occurences": (
        geolocation["geolocation_zip_code_prefix"].duplicated().sum()
    ),
    "exact_duplicate_rows": geolocation.duplicated().sum(),
    "missing_values": geolocation.isna().sum().sum(),
})

geolocation_key_profile

total_rows                         1000163
unique_zip_prefixes                  19015
duplicate_zip_prefix_occurences     981148
exact_duplicate_rows                261831
missing_values                           0
dtype: int64

In [73]:
# Profile geographic variation within each ZIP code prefix
geolocation_prefix_profile = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg(
        record_count=("geolocation_lat", "size"),
        unique_coordinates=("geolocation_lat", "nunique"),
        unique_cities=("geolocation_city", "nunique"),
        unique_states=("geolocation_state", "nunique"),
    )
)

# Summarize geographic consistency
geolocation_variation_summary = pd.Series({
    "total_zip_prefixes": len(geolocation_prefix_profile),
    "prefixes_with_multiple_records": (
        geolocation_prefix_profile["record_count"].gt(1).sum()
    ),
    "prefixes_with_multiple_latitudes": (
        geolocation_prefix_profile["unique_coordinates"].gt(1).sum()
    ),
    "prefixes_with_multiple_cities": (
        geolocation_prefix_profile["unique_cities"].gt(1).sum()
    ),
    "prefixes_with_multiple_states": (
        geolocation_prefix_profile["unique_states"].gt(1).sum()
    ),
    "maximum_records_per_prefix": (
        geolocation_prefix_profile["record_count"].max()
    ),
})

geolocation_variation_summary

total_zip_prefixes                  19015
prefixes_with_multiple_records      17972
prefixes_with_multiple_latitudes    17781
prefixes_with_multiple_cities        8556
prefixes_with_multiple_states           8
maximum_records_per_prefix           1146
dtype: int64

In [74]:
# Validate geographic coordinate ranges
invalid_latitude_mask = (
    ~geolocation["geolocation_lat"].between(-90, 90)
    | geolocation["geolocation_lat"].isna()
)

invalid_longitude_mask = (
    ~geolocation["geolocation_lng"].between(-180, 180)
    | geolocation["geolocation_lng"].isna()
)

# Identify records with invalid coordinates
invalid_coordinates_mask = (
    invalid_latitude_mask | invalid_longitude_mask
)

coordinate_validation = pd.Series({
    "total_records": len(geolocation),
    "invalid_latitudes": invalid_latitude_mask.sum(),
    "invalid_longitude": invalid_longitude_mask.sum(),
    "records_with_invalid_coordinates": invalid_coordinates_mask.sum(),
    "minimum_latitude": geolocation["geolocation_lat"].min(),
    "maximum_latitude": geolocation["geolocation_lat"].max(),
    "minimum_longitude": geolocation["geolocation_lng"].min(),
    "maximum_longitude": geolocation["geolocation_lng"].max(),
})

coordinate_validation

total_records                       1.000163e+06
invalid_latitudes                   0.000000e+00
invalid_longitude                   0.000000e+00
records_with_invalid_coordinates    0.000000e+00
minimum_latitude                   -3.660537e+01
maximum_latitude                    4.506593e+01
minimum_longitude                  -1.014668e+02
maximum_longitude                   1.211054e+02
dtype: float64

In [75]:
# Define approximate geographic bounds for Brazil
brazil_lat_min = -34
brazil_lat_max = 6
brazil_lng_min = -74
brazil_lng_max = -34

# Identify coordinates outside the approximate bounds
outside_brazil_mask = (
    ~geolocation["geolocation_lat"].between(
        brazil_lat_min, brazil_lat_max
    )
    | ~geolocation["geolocation_lng"].between(
        brazil_lng_min, brazil_lng_max
    )
)

# Summarize geographically suspicious records
geographic_plausibility = pd.Series({
    "total_records": len(geolocation),
    "records_outside_brazil_bounds": outside_brazil_mask.sum(),
    "affected_zip_prefixes": (
        geolocation.loc[
            outside_brazil_mask, "geolocation_zip_code_prefix"
        ].nunique()
    ),
    "percentage_outside_bounds": outside_brazil_mask.mean() * 100,
})

geographic_plausibility.round(3)

total_records                    1000163.000
records_outside_brazil_bounds         42.000
affected_zip_prefixes                 21.000
percentage_outside_bounds              0.004
dtype: float64

In [76]:
# Investigate records outside the approximate Brazil bounds
geographic_outliers = geolocation.loc[
    outside_brazil_mask
].copy()

# Sort by ZIP code prefix for easier comparison
geographic_outliers = geographic_outliers.sort_values(
    "geolocation_zip_code_prefix"
)

geographic_outliers.head(15)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [77]:
# Identify ZIP prefizes containing suspicious coordinates
outlier_zip_prefixes = geographic_outliers[
    "geolocation_zip_code_prefix"
].unique()

# Examine all records associated with affected prefixes
affected_geolocation_records = geolocation.loc[
    geolocation["geolocation_zip_code_prefix"].isin(outlier_zip_prefixes)
].copy()

# Compare suspicious and plausible records within each prefix
affected_prefix_summary = (
    affected_geolocation_records.assign(
        outside_bounds=outside_brazil_mask
    )
    .groupby("geolocation_zip_code_prefix")
    .agg(
        total_records=("geolocation_lat", "size"),
        suspicious_records=("outside_bounds", "sum"),
    )
)

affected_prefix_summary["plausible_records"] = (
    affected_prefix_summary["total_records"]
    - affected_prefix_summary["suspicious_records"]
)

affected_prefix_summary

,total_records,suspicious_records,plausible_records
geolocation_zip_code_prefix,,,
18243,1,1,0
28155,5,2,3
28165,4,1,3
28333,7,1,6
28595,10,1,9
29654,4,2,2
35179,85,2,83
45936,35,2,33
46560,2,1,1


In [78]:
# Identify ZIP prefixes associated with multiple states
multi_state_prefixes = geolocation_prefix_profile.index[
    geolocation_prefix_profile["unique_states"] > 1
]

# Retrieve geographic records for those ZIP prefixes
multi_state_records = geolocation.loc[
    geolocation["geolocation_zip_code_prefix"].isin(multi_state_prefixes)
].copy()

# Summarize state assignments within each ZIP prefix
multi_state_summary = (
    multi_state_records.groupby(
        ["geolocation_zip_code_prefix", "geolocation_state"]
    )
    .agg(
        record_count=("geolocation_lat", "size"),
        unique_cities=("geolocation_city", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["geolocation_zip_code_prefix", "record_count"],
        ascending=[True, False],
    )
)

multi_state_summary

,geolocation_zip_code_prefix,geolocation_state,record_count,unique_cities
1,2116,SP,12,2
0,2116,RN,1,1
3,4011,SP,178,2
2,4011,AC,1,1
5,21550,RJ,170,1
4,21550,AC,1,1
7,23056,RJ,60,1
6,23056,AC,1,1
9,72915,GO,40,2
8,72915,DF,1,1


In [79]:
# Identify the most frequently recorded state for each ZIP prefix
majority_states = (
    multi_state_summary
    .drop_duplicates(subset="geolocation_zip_code_prefix")
    [["geolocation_zip_code_prefix", "geolocation_state"]]
    .rename(columns={"geolocation_state": "majority_state"})
)

# Attach the majority state to each geographic record
state_comparison = multi_state_records.merge(
    majority_states,
    on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
)

# Identify records whose state differs from the majority
minority_state_records = state_comparison.loc[
    state_comparison["geolocation_state"]
    != state_comparison["majority_state"]
].copy()

# Display records for geographic inspection
minority_state_records[
    [
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
        "majority_state",
        "geolocation_lat",
        "geolocation_lng",
    ]
].sort_values("geolocation_zip_code_prefix")

,geolocation_zip_code_prefix,geolocation_city,geolocation_state,majority_state,geolocation_lat,geolocation_lng
5,2116,são paulo,RN,SP,-23.515978,-46.582170
69,4011,sao paulo,AC,SP,-23.578707,-46.645779
303,21550,rio de janeiro,AC,RJ,-22.857861,-43.352613
375,23056,rio de janeiro,AC,RJ,-22.919164,-43.611097
430,72915,taguatinga,DF,GO,-12.408760,-46.428287
466,78557,nova brasilandia d'oeste,RO,MT,-11.875142,-55.503020
655,79750,nova andradina,RS,MS,-22.242062,-53.343159
828,80630,balneario de picarras,SC,PR,-26.757371,-48.675738


In [80]:
# Consolidate geolocation data quality findings
geolocation_quality_summary = pd.Series({
    "total_records": len(geolocation),
    "unique_zip_prefixes": (
        geolocation["geolocation_zip_code_prefix"].nunique()
    ),
    "exact_duplicate_rows": geolocation.duplicated().sum(),
    "prefixes_with_multiple_cities": (
        geolocation_prefix_profile["unique_cities"].gt(1).sum()
    ),
    "prefixes_with_multiple_states": (
        geolocation_prefix_profile["unique_states"].gt(1).sum()
    ),
    "minority_state_records": len(minority_state_records),
    "records_outside_approximate_brazil_bounds": (
        outside_brazil_mask.sum()
    ),
    "prefixes_with_out_of_bounds_coordinates": (
        geolocation.loc[
            outside_brazil_mask, "geolocation_zip_code_prefix"
        ].nunique()
    ),
})

geolocation_quality_summary

total_records                                1000163
unique_zip_prefixes                            19015
exact_duplicate_rows                          261831
prefixes_with_multiple_cities                   8556
prefixes_with_multiple_states                      8
minority_state_records                             8
records_outside_approximate_brazil_bounds         42
prefixes_with_out_of_bounds_coordinates           21
dtype: int64

## 7. Sellers Dataset Profiling

In [81]:
# Load the sellers dataset
sellers_path = (
    project_root / "data" / "raw" / "olist_sellers_dataset.csv"
)

sellers = pd.read_csv(sellers_path)

sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [82]:
# inspect row count, column types, and missing values
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


In [83]:
# Validate seller identifiers
seller_key_validation = pd.Series({
    "total_rows": len(sellers),
    "unique_seller_ids": sellers["seller_id"].nunique(),
    "duplicate_seller_ids": sellers["seller_id"].duplicated().sum(),
    "missing_seller_ids": sellers["seller_id"].isna().sum(),
    "exact_duplicate_rows": sellers.duplicated().sum(),
})

seller_key_validation

total_rows              3095
unique_seller_ids       3095
duplicate_seller_ids       0
missing_seller_ids         0
exact_duplicate_rows       0
dtype: int64

In [84]:
# Validate referential integrity between sellers and order items
seller_ids = set(sellers["seller_id"].dropna())
item_seller_ids = set(order_items["seller_id"].dropna())

sellers_without_items = seller_ids - item_seller_ids
items_without_sellers = item_seller_ids - seller_ids

seller_relationship_validation = pd.Series({
    "total_sellers": len(seller_ids),
    "sellers_with_items": len(seller_ids & item_seller_ids),
    "sellers_without_items": len(sellers_without_items),
    "item_seller_ids_without_sellers": len(items_without_sellers),
})

seller_relationship_validation

total_sellers                      3095
sellers_with_items                 3095
sellers_without_items                 0
item_seller_ids_without_sellers       0
dtype: int64

In [85]:
# Validate seller ZIP code coverage against geolocation
seller_zip_prefixes = set(
    sellers["seller_zip_code_prefix"].dropna()
)

geolocation_zip_prefixes = set(
    geolocation["geolocation_zip_code_prefix"].dropna()
)

unmatched_seller_zip_prefixes = (
    seller_zip_prefixes - geolocation_zip_prefixes
)

# Identify seller records without matching geolocation prefixes
sellers_without_geolocation = sellers.loc[
    ~sellers["seller_zip_code_prefix"].isin(geolocation_zip_prefixes)
].copy()

# Summarize geographic coverage
seller_geolocation_validation = pd.Series({
    "total_sellers": len(sellers),
    "unique_seller_zip_prefixes": len(seller_zip_prefixes),
    "matched_seller_zip_prefixes": len(
        seller_zip_prefixes & geolocation_zip_prefixes
    ),
    "unmatched_seller_zip_prefixes": len(unmatched_seller_zip_prefixes),
    "sellers_without_geolocation": len(sellers_without_geolocation),
})

seller_geolocation_validation

total_sellers                    3095
unique_seller_zip_prefixes       2246
matched_seller_zip_prefixes      2239
unmatched_seller_zip_prefixes       7
sellers_without_geolocation         7
dtype: int64

## 8. Products Dataset Profiling

In [86]:
# Load the products dataset
products_path = (
    project_root / "data" / "raw" / "olist_products_dataset.csv"
)

products = pd.read_csv(products_path)
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [87]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [88]:
# Validate product identifiers
product_key_validation = pd.Series({
    "total_rows": len(products),
    "unique_product_ids": products["product_id"].nunique(),
    "duplicate_product_ids": products["product_id"].duplicated().sum(),
    "missing_product_ids": products["product_id"].isna().sum(),
    "exact_duplicate_rows": products.duplicated().sum(),
})

product_key_validation

total_rows               32951
unique_product_ids       32951
duplicate_product_ids        0
missing_product_ids          0
exact_duplicate_rows         0
dtype: int64

In [89]:
# Define product attributes to investigate
product_attribute_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
]

# Profile missing values
product_missing_profile = pd.DataFrame({
    "missing_count": products[product_attribute_columns].isna().sum(),
    "missing_pct": (
        products[product_attribute_columns].isna().mean() * 100
    ),
})

product_missing_profile.round(2)

,missing_count,missing_pct
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85


In [90]:
# Investigate missing attribute patterns
product_missing_mask = products[product_attribute_columns].isna()

product_missing_summary = pd.Series({
    "total_products": len(products),
    "products_with_any_missing_attributes": (
        product_missing_mask.any(axis=1).sum()
    ),
    "products_with_all_attributes_missing": (
        product_missing_mask.all(axis=1).sum()
    ),
    "products_with_partial_missing_attributes": (
        (
            product_missing_mask.any(axis=1)
            & ~product_missing_mask.all(axis=1)
        ).sum()
    ),
})

product_missing_summary

total_products                              32951
products_with_any_missing_attributes          610
products_with_all_attributes_missing          610
products_with_partial_missing_attributes        0
dtype: int64

In [91]:
# Define physical products attributes
product_physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

# Identify products with missing physical attributes
missing_physical_mask = products[product_physical_columns].isna().any(axis=1)

products_missing_physical = products.loc[
    missing_physical_mask
].copy()

products_missing_physical

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [92]:
# Validate physical product measurements
physical_attribute_validation = pd.DataFrame({
    "missing_values": products[product_physical_columns].isna().sum(),
    "zero_values": products[product_physical_columns].eq(0).sum(),
    "negative_values": products[product_physical_columns].lt(0).sum(),
    "minimum": products[product_physical_columns].min(),
    "maximum": products[product_physical_columns].max(),
})

physical_attribute_validation

,missing_values,zero_values,negative_values,minimum,maximum
product_weight_g,2,4,0,0.0,40425.0
product_length_cm,2,0,0,7.0,105.0
product_height_cm,2,0,0,2.0,105.0
product_width_cm,2,0,0,6.0,118.0


In [93]:
# Validate referential integrity between products and order items
product_ids = set(products["product_id"].dropna())
item_product_ids = set(order_items["product_id"].dropna())

products_without_items = product_ids - item_product_ids
items_without_products = item_product_ids - product_ids

product_relationship_validation = pd.Series({
    "total_products": len(product_ids),
    "products_with_items": len(product_ids & item_product_ids),
    "products_without_items": len(products_without_items),
    "item_product_ids_without_products": len(items_without_products),
})

product_relationship_validation

total_products                       32951
products_with_items                  32951
products_without_items                   0
item_product_ids_without_products        0
dtype: int64

In [94]:
# Identify products with missing or zero weight
invalid_weight_mask = (
    products["product_weight_g"].isna()
    | products["product_weight_g"].eq(0)
)

products_invalid_weight = products.loc[
    invalid_weight_mask
].copy()

# Count order item records associated with these products
invalid_weight_impact = (
    products_invalid_weight[
        ["product_id", "product_category_name", "product_weight_g"]
    ]
    .merge(
        order_items.groupby("product_id", as_index=False)
        .agg(order_item_count=("order_id", "size")),
        on="product_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("order_item_count", ascending=False)
)

invalid_weight_impact

,product_id,product_category_name,product_weight_g,order_item_count
4,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,17
5,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0.0,4
3,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0.0,2
0,09ff539a621711667c43eba6a3bd8466,bebes,NaN,1
1,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0.0,1
2,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0.0,1


## 9. Product Category Translation Dataset Profiling

In [95]:
# Load product category translations
category_translation_path = (
    project_root / "data" / "raw" / "product_category_name_translation.csv"
)

category_translation = pd.read_csv(category_translation_path)

category_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [96]:
# Inspect dataset structure
category_translation.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [102]:
# Validate product category translation identifiers
category_translation_validation = pd.Series({
    "total_rows": len(category_translation),
    "unique_portuguese_categories": (
        category_translation["product_category_name"].nunique()
    ),
    "unique_english_categories": (
        category_translation["product_category_name_english"].nunique()
    ),
    "duplicate_portuguese_categories": (
        category_translation["product_category_name"].duplicated().sum()
    ),
    "duplicate_english_categories": (
        category_translation["product_category_name_english"].duplicated().sum()
    ),
    "missing_values": category_translation.isna().sum().sum(),
    "exact_duplicate_rows": category_translation.duplicated().sum(),
})

category_translation_validation

total_rows                         71
unique_portuguese_categories       71
unique_english_categories          71
duplicate_portuguese_categories     0
duplicate_english_categories        0
missing_values                      0
exact_duplicate_rows                0
dtype: int64

In [104]:
# Validate product category tranlation coverage
product_categories = set(
    products["product_category_name"].dropna()
)

translated_categories = set(
    category_translation["product_category_name"].dropna()
)

# Identify categories without translations
untranslated_categories = product_categories - translated_categories

# Identify products affected by missing translations
products_without_translation = products.loc[
    products["product_category_name"].isin(untranslated_categories)
].copy()

# Summarize translation coverage
category_coverage_validation = pd.Series({
    "unique_product_categories": len(product_categories),
    "translated_product_categories": len(
        product_categories & translated_categories
    ),
    "untranslated_product_categories": len(untranslated_categories),
    "products_with_missing_category": (
        products["product_category_name"].isna().sum()
    ),
    "product_with_untranslated_category": len(products_without_translation),
    "unused_translation_categories": len(
        translated_categories - product_categories
    ),
})

category_coverage_validation

unique_product_categories              73
translated_product_categories          71
untranslated_product_categories         2
products_with_missing_category        610
product_with_untranslated_category     13
unused_translation_categories           0
dtype: int64

In [105]:
# Inspect categories without English translations
untranslated_category_summary = (
    products_without_translation
    .groupby("product_category_name")
    .agg(product_count=("product_id", "size"))
    .reset_index()
    .sort_values("product_count", ascending=False)
)

untranslated_category_summary

,product_category_name,product_count
1,portateis_cozinha_e_preparadores_de_alimentos,10
0,pc_gamer,3


In [106]:
# Join products with English category translation
products_translated = products.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one",
    indicator=True,
)

# Validate row preservation and translation coverage
translation_join_validation = pd.Series({
    "products_before_join": len(products),
    "products_after_join": len(products_translated),
    "unique_products_after_join": products_translated["product_id"].nunique(),
    "duplicate_product_ids": products_translated["product_id"].duplicated().sum(),
    "products_with_translation": products_translated["_merge"].eq("both").sum(),
    "products_without_translation": products_translated["_merge"].eq("left_only").sum(),
})

translation_join_validation

products_before_join            32951
products_after_join             32951
unique_products_after_join      32951
duplicate_product_ids               0
products_with_translation       32328
products_without_translation      623
dtype: int64